# Extraccion de variables procesales desde el texto de las actuaciones

La base de actuaciones trae una fila por anotacion del expediente, en
texto libre. Este script la recorre con expresiones regulares y devuelve
una fila por radicado con las fechas de cada etapa, los juzgados, la
medida de aseguramiento, el delito imputado y la sentencia.

Salida: `resultado_variables.parquet`.

In [ ]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

RUTA_ACTUACIONES = Path(
    r"c:\Users\DiDi\Downloads\actuaciones_2006_2026.parquet"
)

N_FILAS = None

COL_RADICADO = "radicado"
COL_ACTUACION = "Actuación"
COL_ANOTACION = "Anotación"
COL_FECHA = "Fecha de Actuación"

COLUMNAS = [COL_ACTUACION, COL_ANOTACION, COL_FECHA, COL_RADICADO]

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [ ]:
if not RUTA_ACTUACIONES.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo:\n  {RUTA_ACTUACIONES}\n"
        "Ajusta RUTA_ACTUACIONES en la celda de configuración."
    )


actuaciones = pd.read_parquet(RUTA_ACTUACIONES)

actuaciones.columns = actuaciones.columns.str.strip()

faltantes = [c for c in COLUMNAS if c not in actuaciones.columns]
if faltantes:
    raise KeyError(
        f"Faltan columnas requeridas: {faltantes}\n"
        f"Columnas disponibles: {actuaciones.columns.tolist()}"
    )


print(f"Filas cargadas: {len(actuaciones):,}")
actuaciones.head()

In [ ]:
def limpiar_texto(texto) -> str:
    if pd.isna(texto):
        return ""

    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower()

    return re.sub(r"\s+", " ", texto).strip()


base = (
    actuaciones if N_FILAS is None else actuaciones.head(N_FILAS)
).copy()

base[COL_ACTUACION] = base[COL_ACTUACION].map(limpiar_texto)
base[COL_ANOTACION] = base[COL_ANOTACION].map(limpiar_texto)
base[COL_FECHA] = pd.to_datetime(base[COL_FECHA], errors="coerce")

base["texto"] = base[COL_ACTUACION] + " " + base[COL_ANOTACION]

print(f"Filas en el subconjunto de trabajo: {len(base):,}")
print(f"Fechas no parseables: {base[COL_FECHA].isna().sum():,}")
base.head()

In [ ]:
PAT_LEG = r"legaliz.*capt"
PAT_IMP = r"formul.*imput|imputacion de cargos"
PAT_MED = r"medida\s+de\s+asegur|impo.*asegur"
PAT_BDET = r"boleta.*deten"
PAT_BLIB = r"boleta.*libert"
PAT_JUICIO_ORAL = r"juicio.*oral"

PAT_ACUSACION = (
    r"escrito\s+de\s+acusacion|"
    r"acusacion\s+escrita|"
    r"audiencia\s+(?:\w+\s+){0,2}?acusacion|"
    r"formul\w*\s+(?:de\s+)?acusacion|"
    r"presenta\w*\s+(?:el\s+)?escrito\s+de\s+acusacion|"
    r"\bacusacion\b"
)

PAT_NO_REALIZADA = (
    r"aplaz\w*|"
    r"reprogram\w*|"
    r"reasign\w*|"
    r"cancel\w*|"
    r"desist\w*|"
    r"se\s+retira\s+la\s+solicitud|"
    r"no\s+se\s+(?:realiz|llev|efectu|instal|celebr|surti)\w*|"
    r"no\s+comparec\w*|"
    r"sin\s+realizar|"
    r"nulidad"
)

PAT_ACUSACION_EXCL = (
    PAT_NO_REALIZADA + "|"
    r"traslado\s+d[eo]l?\s+(?:escrito\s+de\s+)?acusacion|"
    r"copias?\s+d[eo]l?\s+(?:escrito\s+de\s+)?acusacion"
)

PAT_IMP_EXCL = (
    PAT_NO_REALIZADA + "|"
    r"traslado\s+d[eo]\s+(?:la\s+)?imputacion|"
    r"copias?\s+d[eo]\s+(?:la\s+)?imputacion"
)

PAT_PREACUERDO = r"preacuerdo|pre\s+acuerdo"

PAT_DOMICILIARIA = (
    r"domiciliari[ao]|"
    r"arresto\s+domiciliario|"
    r"casa\s+por\s+carcel|"
    r"lugar\s+de\s+residencia|"
    r"en\s+(?:su|el|la)\s+(?:propi[ao]\s+)?"
    r"(?:domicilio|residencia|vivienda|morada|habitacion)|"
    r"en\s+(?:el\s+)?domicilio\s+d[eo]l?\b"
)

PAT_INTRAMURAL = (
    r"intramural|"
    r"establecimiento\s+(?:carcelario|penitenciario)|"
    r"centro\s+de\s+reclusion|"
    r"\bcarcel\b|"
    r"detencion\s+preventiva|"
    r"privativa\s+de\s+la\s+libertad"
)

PAT_IMPONE = r"\bimpone\b|\bimpuso\b|\bimponer\b|\bdecreta\b|\bimposicion\b"

PAT_SUSTITUCION = (
    r"en\s+su\s+lugar|"
    r"en\s+lugar\s+de\s+ello|"
    r"en\s+cambio|"
    r"se\s+sustituye|"
    r"sustituy\w*\s+por|"
    r"se\s+reemplaza|"
    r"se\s+conmuta|"
    r"se\s+cambia\s+por"
)

PAT_NEGADOR = (
    r"\b(?:no|niega|niegan|negar|negada|sin|improcedente|revoca|rechaza|"
    r"abstiene|desiste|retira)\b"
)

PAT_LEG_PRIO = r"legaliza|avala|control de legalidad"
PAT_IMP_PRIO = r"avala|acepta cargos|acepta imputacion|legalidad de imputacion"
PAT_SENT_PRIO = (
    r"\bcondena\b|"
    r"\bcondeno\b|"
    r"\bfallo condenatorio\b|"
    r"\bsentencia condenatoria\b"
)

PAT_SENT = r"individualizacion|\bsentencia\b|\bfallo\b"
PAT_SENT_ANOT = (
    r"(?<!susp )(?<!susp\. )\bcond\b\.?|"
    r"\bcondena\b|"
    r"\bcondeno\b|"
    r"\bfallo condenatorio\b|"
    r"\bsentencia condenatoria\b|"
    r"\babsuelve\b|"
    r"\babsolutoria\b"
)

PAT_ABSOLUCION = (
    r"\babsolutori[oa]\b|"
    r"\babsuelve\b|"
    r"\babsuelto\b|"
    r"\babsolucion\b|"
    r"\bfallo absolutorio\b"
)

PAT_CONDENA = (
    r"(?<!susp )(?<!susp\. )\bcond\b\.?|"
    r"\bfallo condenatorio\b|"
    r"\bsentencia condenatoria\b|"
    r"\bcondena\b|"
    r"\bcondeno\b|"
    r"\bcondenad[oa]\b|"
    r"\bcondenatori[oa]\b"
)

_ABREV_JUZGADO = r"(?:juzgado|juzg|jzgdo|jzgd|jzg|jgdo|jdo|juz|juez)s?"

PAT_JUZGADO_EXT = rf"({_ABREV_JUZGADO}\.?\s*\d{{1,3}})"

PAT_JUZGADO_PREFIJO = rf"^{_ABREV_JUZGADO}\.?\s*"

assert re.compile(PAT_JUZGADO_EXT).groups == 1

In [ ]:
_SEP = r"[\s.,;:/-]+"
_LIGA = r"(?:\w+[\s.,;:/-]+){0,2}?"

_TRAF = r"(?:traf\w*|trfc\w*)"
_FAB = r"(?:fabr\w*|fab\b)"
_PORTE = r"(?:portes?\b|port\b|pte\b|prt\b|portac\w*)"
_TENEN = r"(?:tenenc\w*|tncia\b)"
_ESTUP = r"(?:estup\w*|estp\w*|esup\w*|psicotrop\w*|sicotrop\w*|alucinog\w*)"
_ARMAS = r"(?:armas?\b|arm\b|armament\w*)"
_FUEGO = r"(?:fuego|f\b)"
_HURTO = r"(?:hurtos?\b|hto\b)"
_HOMIC = r"(?:homic\w*|hom\b)"
_LESION = r"(?:lesion\w*|lesn\w*)"
_VIOL = r"(?:violen\w*|viol\b)"
_INTRAF = r"(?:intrafam\w*|intraf\w*)"
_MENOR = r"(?:menor\w*|men\b)"
_CATORCE = r"(?:catorce|14)"
_DIECIOCHO = r"(?:dieciocho|18)"
_PERSONA_PROT = r"person\w*" + _SEP + r"proteg\w*"
_ACCESO_CARNAL = r"acces\w*" + _SEP + r"carnal"
_ACTO_SEXUAL = r"act\w*" + _SEP + r"sexual\w*"
_PROSTIT = r"prostit\w*"
_AGRAV = r"agravac\w*"

DELITOS = [
    (r"abus\w*" + _SEP + r"de" + _SEP + r"confianz\w*" + _SEP + r"calificad[oa]",
     "Artículo 250. Abuso de confianza calificado"),
    (_HURTO + _SEP + r"calificad[oa]", "Artículo 240. Hurto calificado"),
    (_HURTO + _SEP + r"agravad[oa]", "Artículo 239. Hurto"),
    (r"abus\w*" + _SEP + r"de" + _SEP + r"confianz\w*", "Artículo 249. Abuso de confianza"),
    (r"extors\w*", "Artículo 244. Extorsión"),
    (r"estaf\w*", "Artículo 246. Estafa"),
    (_HURTO, "Artículo 239. Hurto"),

    (r"circunstanc\w*" + _SEP + r"de" + _SEP + _AGRAV + _SEP + r"punitiv\w*"
     + _SEP + r"del" + _SEP + r"feminicid\w*",
     "Artículo 104B. Circunstancias de agravación punitiva del feminicidio"),
    (_HOMIC + _SEP + r"preterinten\w*", "Artículo 105. Homicidio preterintencional"),
    (_HOMIC + _SEP + r"por" + _SEP + r"piedad", "Artículo 106. Homicidio por piedad"),
    (_HOMIC + _SEP + r"culpos\w*", "otros"),
    (_HOMIC + _SEP + r"en" + _SEP + _PERSONA_PROT, "Artículo 135. Homicidio en persona protegida"),
    (_HOMIC + _SEP + r"agravad[oa]", "Artículo 103. Homicidio"),
    (r"(?:feminicid\w*|femin\b)", "Artículo 104A. Feminicidio"),
    (_HOMIC, "Artículo 103. Homicidio"),
    (_LESION + _SEP + r"(?:personal\w*" + _SEP + r")?agravad[oa]s?", "Artículo 111. Lesiones (Agravadas)"),
    (_LESION + _SEP + r"personal\w*", "otros"),
    (_LESION, "otros"),

    (r"muert\w*" + _SEP + r"de" + _SEP + r"hijo" + _SEP + r"fruto" + _SEP
     + r"de" + _SEP + _ACCESO_CARNAL + r".{0,100}?"
     + r"(?:inseminac\w*|transferenc\w*)" + r".{0,100}?no" + _SEP + r"consentid[ao]s?",
     "Artículo 108. Muerte de hijo fruto de acceso carnal violento, abusivo, o de inseminación artificial o transferencia de ovulo fecundado no consentidas"),
    (r"abandon\w*" + _SEP + r"de" + _SEP + r"hijo" + _SEP + r"fruto" + _SEP
     + r"de" + _SEP + _ACCESO_CARNAL + r".{0,100}?"
     + r"(?:inseminac\w*|transferenc\w*)" + r".{0,100}?no" + _SEP + r"consentid[ao]s?",
     "Artículo 128. Abandono de hijo fruto de acceso carnal violento, abusivo, o de inseminación artificial o transferencia de ovulo fecundado no consentidas"),

    (_ACCESO_CARNAL + _SEP + r"violent\w*" + _SEP + r"en" + _SEP + _PERSONA_PROT,
     "Artículo 138. Acceso carnal violento en persona protegida"),
    (_ACTO_SEXUAL + _SEP + r"violent\w*" + _SEP + r"en" + _SEP + _PERSONA_PROT
     + r".{0,30}?" + _MENOR + _SEP + r"de" + _SEP + _CATORCE,
     "Artículo 139A. Actos sexuales violentos en persona protegida menor de catorce años"),
    (_ACTO_SEXUAL + _SEP + r"violent\w*" + _SEP + r"en" + _SEP + _PERSONA_PROT,
     "Artículo 139. Actos sexuales violentos en persona protegida"),
    (_ACCESO_CARNAL + _SEP + r"abusiv\w*" + _SEP + r"con" + _SEP + _MENOR
     + _SEP + r"de" + _SEP + _CATORCE,
     "Artículo 208. Acceso carnal abusivo con menor de catorce años"),
    (_ACTO_SEXUAL + _SEP + r"con" + _SEP + _MENOR + _SEP + r"de" + _SEP + _CATORCE,
     "Artículo 209. Actos sexuales con menor de catorce años"),
    (_ACCESO_CARNAL + _SEP + r"violent\w*", "Artículo 205. Acceso carnal violento"),
    (_ACCESO_CARNAL + _SEP + r"abusiv\w*", "otros"),
    (_ACTO_SEXUAL + _SEP + r"violent\w*", "Artículo 206. Acto sexual violento"),
    (r"acos\w*" + _SEP + r"sexual\w*", "Artículo 210A. Acoso sexual"),

    (r"trat\w*" + _SEP + r"de" + _SEP + r"person\w*" + _SEP + r"en" + _SEP
     + _PERSONA_PROT + r".{0,60}?explotac\w*" + _SEP + r"sexual\w*",
     "Artículo 141B. Trata de personas en persona protegida con fines de explotación sexual"),
    (r"esclavit\w*" + _SEP + r"sexual\w*" + _SEP + r"en" + _SEP + _PERSONA_PROT,
     "Artículo 141A. Esclavitud sexual en persona protegida"),
    (_PROSTIT + _SEP + r"forzad[ao]" + _SEP + r"en" + _SEP + _PERSONA_PROT,
     "Artículo 141. Prostitución forzada en persona protegida"),
    (r"proxenet\w*" + _SEP + r"con" + _SEP + _MENOR + _SEP + r"de" + _SEP + r"edad",
     "Artículo 213A. Proxenetismo con menor de edad"),
    (r"constren\w*" + _SEP + r"a" + _SEP + r"la" + _SEP + _PROSTIT,
     "Artículo 214. Constreñimiento a la prostitución"),
    (r"induc\w*" + _SEP + r"a" + _SEP + r"la" + _SEP + _PROSTIT,
     "Artículo 213. Inducción a la prostitución"),
    (r"estimul\w*" + _SEP + r"a" + _SEP + r"la" + _SEP + _PROSTIT + _SEP
     + r"de" + _SEP + _MENOR,
     "Artículo 217. Estímulo a la prostitución de menores"),
    (r"pornograf\w*" + _SEP + _LIGA + _MENOR + r".{0,20}?" + _DIECIOCHO,
     "Artículo 218. Pornografía con personas menores a 18 años"),
    (r"utilizac\w*" + _SEP + _LIGA + r"medi\w*" + _SEP + r"de" + _SEP
     + r"comunicac\w*" + r".{0,100}?activid\w*" + _SEP + r"sexual\w*"
     + r".{0,100}?" + _MENOR + r".{0,20}?" + _DIECIOCHO,
     "Artículo 219A. Utilización o facilitación de medios de comunicación para ofrecer actividades sexuales con personas menores de 18 años"),
    (_PROSTIT, "otros"),

    (r"maltrat\w*" + _SEP + r"mediante" + _SEP + r"restric\w*" + _SEP + r"a"
     + _SEP + r"la" + _SEP + r"libertad" + _SEP + r"fisic\w*",
     "Artículo 230. Maltrato mediante restricción a la libertad física"),
    (r"ejercic\w*" + _SEP + r"arbitrar\w*" + _SEP + r"de" + _SEP + r"la"
     + _SEP + r"custod\w*" + r".{0,40}?hijo" + _SEP + _MENOR,
     "Artículo 230A. Ejercicio arbitrario de la custodia de hijo menor de edad"),
    (r"maltrat\w*" + _SEP + r"por" + _SEP + r"descuid\w*" + r".{0,30}?negligenc\w*"
     + r".{0,30}?abandon\w*" + r".{0,50}?(?:mayor" + _SEP + r"de" + _SEP
     + r"60|person\w*" + _SEP + r"mayor)",
     "Artículo 229A. Maltrato por descuido, negligencia o abandono en persona mayor de 60 años"),
    (r"(?:" + _VIOL + _SEP + _INTRAF + r"|\bvif\b)", "Artículo 229. Violencia intrafamiliar"),
    (r"maltrat\w*", "otros"),

    (r"asesoram\w*" + _SEP + r"a" + _SEP + r"grup\w*" + _SEP + r"delict\w*"
     + r".{0,60}?grup\w*" + _SEP + r"armad\w*",
     "Artículo 340A. Asesoramiento a grupos delictivos organizados y grupos armados organizados"),
    (r"conciert\w*" + _SEP + r"(?:para" + _SEP + r")?delinq\w*",
     "Artículo 340. Concierto para delinquir"),
    (r"entrenam\w*" + _SEP + r"para" + _SEP + r"activid\w*" + _SEP + r"ilicit\w*",
     "Artículo 341. Entrenamiento para actividades ilícitas"),

    (r"administrac\w*" + _SEP + r"de" + _SEP + r"recurs\w*"
     + r".{0,60}?activid\w*" + _SEP + r"terrorist\w*",
     "Artículo 345. Administración de recursos relacionados con actividades terroristas"),
    (r"utilizac\w*" + _SEP + r"ilegal" + _SEP + r"de" + _SEP + r"uniform\w*",
     "Artículo 346. Utilización ilegal de uniformes e insignias"),
    (r"incitac\w*" + _SEP + r"a" + _SEP + r"la" + _SEP + r"comis\w*" + _SEP
     + r"de" + _SEP + r"delit\w*" + _SEP + r"militar\w*",
     "Artículo 349. Incitación a la comisión de delitos militares"),
    (r"instigac\w*" + _SEP + r"a" + _SEP + r"delinq\w*", "Artículo 348. Instigación a delinquir"),
    (r"terroris\w*", "Artículo 343. Terrorismo"),
    (r"amenaz\w*", "Artículo 347. Amenazas"),

    (_FAB + r".{0,20}?" + _TRAF + r".{0,30}?" + _PORTE + _SEP + r"de" + _SEP
     + _ARMAS + r".{0,80}?(?:uso" + _SEP + r"restring\w*|uso" + _SEP
     + r"privativ\w*|fuerz\w*" + _SEP + r"armad\w*|explosiv\w*)",
     "Artículo 366. Fabricación, tráfico y porte de armas, municiones de uso restringido, de uso privativo de las fuerzas armadas o explosivos"),
    (_FAB + r".{0,30}?(?:" + _TRAF + r"|" + _PORTE + r"|" + _TENEN
     + r").{0,50}?" + _ARMAS + _SEP + r"de" + _SEP + _FUEGO,
     "Artículo 365. Fabricación, tráfico o tenencia de armas de fuego, accesorios, partes o municiones"),
    (_PORTE + _SEP + r"ilegal" + _SEP + r"de" + _SEP + _ARMAS + r"(?:" + _SEP
     + r"de" + _SEP + _FUEGO + r")?",
     "Artículo 365. Fabricación, tráfico o tenencia de armas de fuego, accesorios, partes o municiones"),
    (_FAB + r".{0,20}?" + _TRAF + r".{0,20}?" + _PORTE + _SEP + r"de" + _SEP + _ARMAS,
     "Artículo 365. Fabricación, tráfico o tenencia de armas de fuego, accesorios, partes o municiones"),
    ("(?:" + _PORTE + r"|" + _TENEN + r")" + _SEP + r"de" + _SEP + _ARMAS
     + _SEP + r"de" + _SEP + _FUEGO,
     "Artículo 365. Fabricación, tráfico o tenencia de armas de fuego, accesorios, partes o municiones"),

    (_TRAF + r".{0,20}?" + _FAB + r".{0,20}?" + _PORTE + _SEP + r"de" + _SEP + _ESTUP,
     "Artículo 376. Tráfico, fabricación o porte de estupefacientes"),
    (_FAB + r".{0,20}?" + _TRAF + r".{0,20}?" + _PORTE + _SEP + r"de" + _SEP + _ESTUP,
     "Artículo 376. Tráfico, fabricación o porte de estupefacientes"),
    (_TRAF + r".{0,20}?" + _PORTE + _SEP + r"de" + _SEP + _ESTUP,
     "Artículo 376. Tráfico, fabricación o porte de estupefacientes"),
    (r"conserv\w*" + _SEP + r"o" + _SEP + r"financiac\w*" + _SEP + r"de" + _SEP
     + r"plantac\w*",
     "otros"),
    (_TRAF + _SEP + r"de" + _SEP + _ESTUP, "Artículo 376. Tráfico, fabricación o porte de estupefacientes"),
    (_PORTE + _SEP + r"de" + _SEP + _ESTUP, "Artículo 376. Tráfico, fabricación o porte de estupefacientes"),
    (_TRAF + _SEP + _ESTUP, "Artículo 376. Tráfico, fabricación o porte de estupefacientes"),
    (_PORTE + _SEP + _ESTUP, "Artículo 376. Tráfico, fabricación o porte de estupefacientes"),

    (r"circunstanc\w*" + _SEP + r"de" + _SEP + _AGRAV + _SEP + r"punitiv\w*",
     "Artículo 119. Circunstancias de agravación punitiva"),
]

_DELITOS = [(re.compile(p), etiqueta) for p, etiqueta in DELITOS]

_GATE_DELITO = re.compile(
    r"hurt|hto\b|abus|extors|estaf|homic|\bhom\b|lesion|femin|acces|sexual|"
    r"acos|prostit|proxenet|constren|induc|estimul|pornograf|maltrat|violen|"
    r"\bviol\b|\bvif\b|conciert|delinq|entrenam|asesoram|terroris|amenaz|"
    r"instigac|incitac|uniform|fabr|\bfab\b|traf|port|\bpte\b|tenenc|"
    r"estup|esup|psicotrop|sicotrop|alucinog|arma|circunstanc|muert|abandon|"
    r"trata|esclavit|conserv|plantac|militar"
)


def clasificar_delitos(texto: str) -> list:
    if not texto or not _GATE_DELITO.search(texto):
        return []

    ocupados, encontrados = [], []

    for patron, etiqueta in _DELITOS:
        for m in patron.finditer(texto):
            if any(m.start() < fin and ini < m.end() for ini, fin in ocupados):
                continue

            ocupados.append((m.start(), m.end()))
            encontrados.append((m.start(), etiqueta))

    encontrados.sort(key=lambda x: x[0])

    salida = []
    for _, etiqueta in encontrados:
        if etiqueta not in salida:
            salida.append(etiqueta)

    return salida


def clasificar_delito(texto: str):
    delitos = clasificar_delitos(texto)

    return delitos[0] if delitos else np.nan


_OTROS_DELITOS = [
    r"receptac\w*",
    r"falsedad",
    r"document\w*\s+fals\w*",
    r"fraude",
    r"secuestr\w*",
    r"desaparic\w*\s+forzad\w*",
    r"tortur\w*",
    r"rebeli\w*",
    r"sedici\w*",
    r"asonad\w*",
    r"peculad\w*",
    r"cohech\w*",
    r"concusi\w*",
    r"prevaricat\w*",
    r"enriquecim\w*\s+ilicit\w*",
    r"lavad\w*\s+de\s+activ\w*",
    r"testaferrat\w*",
    r"contraband\w*",
    r"evasi\w*",
    r"fuga\s+de\s+pres\w*",
    r"inasistenc\w*\s+alimentar\w*",
    r"dano\s+en\s+bien\s+ajen\w*",
    r"invasi\w*\s+de\s+tierr\w*",
    r"usurpac\w*",
    r"constrenim\w*",
    r"injuri\w*",
    r"calumni\w*",
    r"fals\w*\s+testimoni\w*",
    r"obstrucc\w*\s+a\s+la\s+justic\w*",
    r"acces\w*\s+abusiv\w*\s+a\s+un?\s+sistem\w*",
    r"violac\w*\s+de\s+habitac\w*",
    r"abus\w*\s+de\s+autorid\w*",
    r"aborto",
    r"bigami\w*",
]

PAT_MENCION_DELITO = (
    r"\bdelit\w*|"
    r"conduct\w*\s+punibl\w*|"
    r"\bpunible\b|"
    r"tipo\s+penal|"
    + "|".join(_OTROS_DELITOS)
)

print(f"Delitos configurados: {len(DELITOS)}")

for _t in [
    "coautores respon de trafico, fab o pte de estupe a la pena de 38 mes",
    "x el delito de porte ilegal de armas de fuego de defensa personal",
    "preacuerdo. traf. estup. jdo 13 ccto.",
    "condena por hurto calificado y agravado",
    "condena por hurto calificado, porte ilegal de armas y amenazas",
    "condena x receptacion agravada",
    "se lee el fallo condenatorio en audiencia",
    "imputa viol. intraf. en contra de",
]:
    print(f"  {clasificar_delitos(_t)!s:<62} <- {_t[:42]}")


print(f"\nEtiquetas distintas: {len({e for _, e in DELITOS})}")

In [ ]:
_NEG_MEDIDA = [
    "niega la medida",
    "niega medida",
    "niega la medida de aseguramiento",
    "niega medida de aseguramiento",
    "se niega la medida",
    "se niega medida",
    "se niega la medida de aseguramiento",
    "se niega medida de aseguramiento",
    "medida de aseguramiento negada",
    "medida negada",
    "niega solicitud de medida",
    "niega la solicitud de medida",
    "niega solicitud de medida de aseguramiento",
    "niega la solicitud de medida de aseguramiento",
    "se niega solicitud de medida",
    "se niega la solicitud de medida",
    "se niega solicitud de medida de aseguramiento",
    "se niega la solicitud de medida de aseguramiento",
    "no impone medida",
    "no impone la medida",
    "no impone medida de aseguramiento",
    "no impone la medida de aseguramiento",
    "no se impone medida",
    "no se impone la medida",
    "no se impone medida de aseguramiento",
    "no se impone la medida de aseguramiento",
    "sin imposicion de medida",
    "sin imposicion de medida de aseguramiento",
    "no hay lugar a imponer medida",
    "no hay lugar a imponer medida de aseguramiento",
    "no procede medida",
    "no procede la medida",
    "no procede medida de aseguramiento",
    "no procede la medida de aseguramiento",
    "improcedente la medida",
    "improcedente medida de aseguramiento",
    "se retira solicitud",
    "se retira la solicitud",
    "se retira solicitud de medida",
    "se retira la solicitud de medida",
    "se retira solicitud de medida de aseguramiento",
    "se retira la solicitud de medida de aseguramiento",
    "retira solicitud",
    "retira la solicitud",
    "retira solicitud de medida",
    "retira la solicitud de medida",
    "retira solicitud de medida de aseguramiento",
    "retira la solicitud de medida de aseguramiento",
    "retiro solicitud",
    "retiro de solicitud",
    "retiro de la solicitud",
    "retiro solicitud de medida",
    "retiro de solicitud de medida",
    "retiro de la solicitud de medida",
    "desiste de la solicitud",
    "desiste solicitud",
    "desiste de solicitar medida",
    "desiste de la medida",
    "desiste de medida de aseguramiento",
    "desistimiento de la solicitud",
    "desistimiento de solicitud",
    "desistimiento de la medida",
    "desistimiento de medida de aseguramiento",
    "se abstiene",
    "se abstiene de imponer",
    "se abstiene de imponer medida",
    "se abstiene de imponer medida de aseguramiento",
    "se abstiene de decretar",
    "se abstiene de decretar medida",
    "se abstiene de decretar medida de aseguramiento",
    "no solicita medida",
    "no solicita medida de aseguramiento",
    "no se solicita medida",
    "no se solicita medida de aseguramiento",
    "fiscalia no solicita medida",
    "fiscal no solicita medida",
    "fiscalia se abstiene de solicitar medida",
    "no avala",
    "no avala la solicitud",
    "no avala medida",
    "no avala la medida",
    "no accede",
    "no accede a la solicitud",
    "no accede a solicitud de medida",
    "no accede a la medida",
    "no accede a medida de aseguramiento",
    "no concede",
    "no concede la medida",
    "no concede medida de aseguramiento",
    "rechaza la solicitud",
    "rechaza solicitud de medida",
    "rechaza la medida",
    "rechaza medida de aseguramiento",
    "no decreta medida",
    "no decreta la medida",
    "no decreta medida de aseguramiento",
    "no otorga",
    "otorga libertad",
    "se otorga libertad",
    "concede libertad",
    "se concede libertad",
    "ordena libertad",
    "ordena la libertad",
    "dispone libertad",
    "dispone la libertad",
    "queda en libertad",
    "deja en libertad",
    "se deja en libertad",
    "libertad inmediata",
    "ordena libertad inmediata",
]

_NO_INTRAMURAL = [
    "detencion domiciliaria",
    "medida domiciliaria",
    "domiciliaria",
    "no privitiva",
    "no privativa",
    "medida no privativa",
    "medida no privativa de la libertad",
]


def _lista_a_patron(palabras) -> str:
    alternativas = "|".join(
        re.escape(p) for p in sorted(set(palabras), key=len, reverse=True)
    )
    return rf"(?<!\w)(?:{alternativas})(?!\w)"


PAT_MEDIDA_NEG = _lista_a_patron(_NEG_MEDIDA)
PAT_NO_INTRAMURAL = _lista_a_patron(_NO_INTRAMURAL)

print(f"Expresiones negativas: {len(_NEG_MEDIDA)}")
print(f"Expresiones no intramurales: {len(_NO_INTRAMURAL)}")

In [ ]:
def _texto(s: pd.Series) -> pd.Series:
    return s.astype("object").fillna("").astype(str)


def extraer_juzgado_serie(s: pd.Series) -> pd.Series:
    result = _texto(s).str.extract(PAT_JUZGADO_EXT, expand=False)

    return (
        result
        .str.replace(PAT_JUZGADO_PREFIJO, "juzgado ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )


def extraer_delitos_serie(s: pd.Series) -> pd.Series:
    return _texto(s).map(clasificar_delitos)


def extraer_delito_serie(s: pd.Series) -> pd.Series:
    return _texto(s).map(clasificar_delito).astype("object")


def esta_afirmado(texto: str, patron: str, ventana: int = 50) -> bool:
    for m in re.finditer(patron, texto):
        previo = texto[max(0, m.start() - ventana): m.start()].split(".")[-1]
        if not re.search(PAT_NEGADOR, previo):
            return True

    return False


def segmento_decisorio(texto: str) -> str:
    ultimo = None
    for m in re.finditer(PAT_SUSTITUCION, texto):
        ultimo = m

    return texto[ultimo.end():] if ultimo else texto


def _primero_no_nulo(s: pd.Series):
    s = s.dropna()
    return s.iloc[0] if len(s) else np.nan


def _primera_lista_no_vacia(valores) -> list:
    for v in valores:
        if isinstance(v, list) and v:
            return v

    return []


def _vacio(columnas: dict) -> pd.DataFrame:
    return pd.DataFrame({c: pd.Series(dtype=d) for c, d in columnas.items()})


DTYPE_RAD = base[COL_RADICADO].dtype


def fecha_por_radicado(df: pd.DataFrame, nombre: str, como="min") -> pd.DataFrame:
    if df.empty:
        return _vacio({COL_RADICADO: DTYPE_RAD, nombre: "datetime64[ns]"})

    serie = df.groupby(COL_RADICADO, sort=False)[COL_FECHA]
    fechas = (serie.min() if como == "min" else serie.max()).reset_index()

    return fechas.rename(columns={COL_FECHA: nombre})


def primera_fila_por_radicado(df: pd.DataFrame, columnas: list) -> pd.DataFrame:
    if df.empty:
        return _vacio(
            {COL_RADICADO: DTYPE_RAD, COL_FECHA: "datetime64[ns]"}
            | {c: "object" for c in columnas}
        )

    return (
        df.sort_values(
            [COL_RADICADO, "priority", COL_FECHA],
            ascending=[True, False, True],
        )
        .groupby(COL_RADICADO, sort=False)[[COL_FECHA] + columnas]
        .first()
        .reset_index()
    )

In [ ]:
print("Computando flags ...")

txt = _texto(base["texto"])

base["flag_legalizacion"] = txt.str.contains(PAT_LEG, na=False)
base["flag_imputacion"] = txt.str.contains(PAT_IMP, na=False) & ~txt.str.contains(
    PAT_IMP_EXCL, na=False
)
base["flag_medida"] = txt.str.contains(PAT_MED, na=False)
base["flag_boleta_det"] = txt.str.contains(PAT_BDET, na=False)
base["flag_boleta_lib"] = txt.str.contains(PAT_BLIB, na=False)
base["flag_juicio"] = txt.str.contains(PAT_JUICIO_ORAL, na=False)
base["flag_acusacion"] = txt.str.contains(PAT_ACUSACION, na=False) & ~txt.str.contains(
    PAT_ACUSACION_EXCL, na=False
)
base["flag_preacuerdo"] = txt.str.contains(PAT_PREACUERDO, na=False)

base["flag_sentencia"] = txt.str.contains(PAT_SENT, na=False) | txt.str.contains(
    PAT_SENT_ANOT, na=False
)

print(
    base[
        [
            "flag_legalizacion",
            "flag_imputacion",
            "flag_medida",
            "flag_boleta_det",
            "flag_boleta_lib",
            "flag_acusacion",
            "flag_juicio",
            "flag_preacuerdo",
            "flag_sentencia",
        ]
    ].sum()
)

In [ ]:
resultado = pd.DataFrame(
    {COL_RADICADO: base[COL_RADICADO].dropna().unique()}
)

print(f"Procesos únicos: {len(resultado):,}")

In [ ]:
print("Extrayendo legalización ...")

leg = base[base["flag_legalizacion"]].copy()
leg["priority"] = _texto(leg["texto"]).str.contains(PAT_LEG_PRIO, na=False).astype(int)

leg_first = primera_fila_por_radicado(leg, ["texto"])
leg_first["JuzgadoCaptura"] = extraer_juzgado_serie(leg_first["texto"])

leg_first = leg_first.rename(columns={COL_FECHA: "FechaLegalizacion"})[
    [COL_RADICADO, "FechaLegalizacion", "JuzgadoCaptura"]
]

resultado = resultado.merge(leg_first, on=COL_RADICADO, how="left")

In [ ]:
print("Extrayendo juicio oral ...")

juicio = fecha_por_radicado(base[base["flag_juicio"]], "FechaJuicioOral", "min")
resultado = resultado.merge(juicio, on=COL_RADICADO, how="left")

print("Extrayendo imputación ...")

imp = base[base["flag_imputacion"]].copy()
imp["priority"] = _texto(imp["texto"]).str.contains(PAT_IMP_PRIO, na=False).astype(int)

imp_first = primera_fila_por_radicado(imp, [])[[COL_RADICADO, COL_FECHA]].rename(
    columns={COL_FECHA: "FechaImputacion"}
)

resultado = resultado.merge(imp_first, on=COL_RADICADO, how="left")

print("Extrayendo acusación ...")

acu = fecha_por_radicado(base[base["flag_acusacion"]], "FechaAcusacion", "min")
resultado = resultado.merge(acu, on=COL_RADICADO, how="left")

print(f"Procesos con fecha de acusación: {resultado['FechaAcusacion'].notna().sum():,}")

In [ ]:
print("Extrayendo medida de aseguramiento ...")

med = base[base["flag_medida"]].copy()

COLS_MED = {
    COL_RADICADO: DTYPE_RAD,
    "TipoMedida": "int64",
    "FechaMedida": "datetime64[ns]",
}

if med.empty:
    med_result = _vacio(COLS_MED)
else:
    med_texto = (
        med.sort_values(COL_FECHA)
        .groupby(COL_RADICADO, sort=False)["texto"]
        .agg(lambda x: " ".join(x.astype(str)))
        .reset_index()
        .rename(columns={"texto": "texto_med"})
    )

    tm = _texto(med_texto["texto_med"])

    hay_negativa = tm.str.contains(PAT_MEDIDA_NEG, na=False)

    tm_modalidad = tm.map(segmento_decisorio)

    es_domiciliaria = tm_modalidad.map(lambda t: esta_afirmado(t, PAT_DOMICILIARIA))
    es_intramural = tm_modalidad.map(lambda t: esta_afirmado(t, PAT_INTRAMURAL))

    es_imposicion = tm.str.contains(PAT_IMPONE, na=False) & ~hay_negativa

    med_texto["TipoMedida"] = np.select(
        [es_domiciliaria, es_intramural, es_imposicion],
        [2, 1, 1],
        default=0,
    )

    fecha_med = fecha_por_radicado(med, "FechaMedida", "min")

    med_result = med_texto[[COL_RADICADO, "TipoMedida"]].merge(
        fecha_med, on=COL_RADICADO, how="left"
    )


resultado = resultado.merge(med_result, on=COL_RADICADO, how="left")

resultado["TipoMedida"] = resultado["TipoMedida"].fillna(0).astype(int)

resultado.loc[resultado["TipoMedida"].eq(0), "FechaMedida"] = pd.NaT

In [ ]:
print("Extrayendo boletas ...")

fbd = fecha_por_radicado(
    base[base["flag_boleta_det"]], "FechaBoletaDetencion", "min"
)
fbl = fecha_por_radicado(
    base[base["flag_boleta_lib"]], "FechaBoletaSalida", "max"
)

resultado = resultado.merge(fbd, on=COL_RADICADO, how="left")
resultado = resultado.merge(fbl, on=COL_RADICADO, how="left")

resultado["BoletaSalida"] = resultado["FechaBoletaSalida"].notna().astype(int)

In [ ]:
print("Marcando preacuerdos ...")

preacuerdo = (
    base.groupby(COL_RADICADO, sort=False)["flag_preacuerdo"]
    .any()
    .astype(int)
    .reset_index()
    .rename(columns={"flag_preacuerdo": "Preacuerdo"})
)

resultado = resultado.merge(preacuerdo, on=COL_RADICADO, how="left")
resultado["Preacuerdo"] = resultado["Preacuerdo"].fillna(0).astype(int)

In [ ]:
print("Extrayendo sentencia ...")

PAT_SOLO_MENCION = (
    r"(?:solicit|pide|peticion|impetra)\w*(?:\s+\S+){0,4}?(?:condena|sentencia)|"
    r"copias?\s+de\s+la\s+sentencia|"
    r"notificacion\s+de\s+la\s+sentencia|"
    r"traslado\s+de\s+la\s+sentencia|"
    r"apelacion\s+(?:contra|de)\s+la\s+sentencia"
)

COLS_SENT = {
    COL_RADICADO: DTYPE_RAD,
    "FechaSentencia": "datetime64[ns]",
    "TieneSentencia": "int64",
    "Sentencia": "int64",
    "TipoSentencia": "int64",
    "JuzgadoSentencia": "object",
    "DelitoSentencia": "object",
}

sent = base[base["flag_sentencia"]].copy()

if sent.empty:
    sent_result = _vacio(COLS_SENT)
else:
    ts = _texto(sent["texto"])

    sent["JuzgadoFila"] = extraer_juzgado_serie(sent["texto"])
    sent["DelitosFila"] = extraer_delitos_serie(sent["texto"])

    menciona_delito = ts.str.contains(PAT_MENCION_DELITO, na=False)
    sent["DelitosFila"] = [
        delitos if delitos else (["otros"] if menciona else [])
        for delitos, menciona in zip(sent["DelitosFila"], menciona_delito)
    ]

    es_abs = ts.str.contains(PAT_ABSOLUCION, na=False)
    es_cond = ts.str.contains(PAT_CONDENA, na=False)

    sent["SentidoFila"] = np.where(es_abs, 0.0, np.where(es_cond, 1.0, np.nan))

    sent["score"] = (
        4 * (es_abs | ts.str.contains(PAT_SENT_PRIO, na=False)).astype(int)
        + 2 * ts.str.contains(PAT_SENT, na=False).astype(int)
        + 1 * sent["DelitosFila"].map(len).gt(0).astype(int)
        - 6 * ts.str.contains(PAT_SOLO_MENCION, na=False).astype(int)
    )

    sent = sent.sort_values(
        [COL_RADICADO, "score", COL_FECHA], ascending=[True, False, True]
    )

    sent_result = (
        sent.groupby(COL_RADICADO, sort=False)
        .agg(
            FechaSentencia=(COL_FECHA, "first"),
            SentidoSentencia=("SentidoFila", _primero_no_nulo),
            JuzgadoSentencia=("JuzgadoFila", _primero_no_nulo),
        )
        .reset_index()
    )

    delitos_por_rad = {
        rad: _primera_lista_no_vacia(grupo)
        for rad, grupo in sent.groupby(COL_RADICADO, sort=False)["DelitosFila"]
    }
    sent_result["DelitoSentencia"] = sent_result[COL_RADICADO].map(delitos_por_rad)

    sent_result["TieneSentencia"] = 1
    sentido = sent_result.pop("SentidoSentencia")
    sent_result["Sentencia"] = sentido.fillna(0).astype(int)

    sent_result["TipoSentencia"] = np.select(
        [sentido.eq(1), sentido.eq(0)], [1, 2], default=0
    ).astype(int)

    sent_result["DelitoSentencia"] = [
        delitos if sentido == 1 else []
        for delitos, sentido in zip(
            sent_result["DelitoSentencia"], sent_result["Sentencia"]
        )
    ]

    sent_result = sent_result[list(COLS_SENT)]


resultado = resultado.merge(sent_result, on=COL_RADICADO, how="left")

resultado["Sentencia"] = resultado["Sentencia"].fillna(0).astype(int)
resultado["TieneSentencia"] = resultado["TieneSentencia"].fillna(0).astype(int)
resultado["TipoSentencia"] = resultado["TipoSentencia"].fillna(0).astype(int)

In [ ]:
ORDEN = [
    COL_RADICADO,
    "FechaLegalizacion",
    "JuzgadoCaptura",
    "FechaImputacion",
    "FechaMedida",
    "TipoMedida",
    "FechaBoletaDetencion",
    "FechaBoletaSalida",
    "BoletaSalida",
    "FechaAcusacion",
    "FechaJuicioOral",
    "Preacuerdo",
    "FechaSentencia",
    "TieneSentencia",
    "Sentencia",
    "TipoSentencia",
    "JuzgadoSentencia",
    "DelitoSentencia",
]
resultado = resultado[ORDEN]

n_procesos = len(resultado)
n_multiples = resultado["DelitoSentencia"].map(
    lambda d: len(d) if isinstance(d, list) else 0
).gt(1).sum()

resultado = resultado.explode("DelitoSentencia", ignore_index=True)

print(f"Procesos: {n_procesos:,}")
print(f"Procesos con más de un delito: {n_multiples:,}")
print(f"Filas del resultado (una por delito): {len(resultado):,}")
print("\nCobertura de cada variable (% no nulo):")
print((resultado.notna().mean() * 100).round(1).to_string())

print("\nTipo de medida (0 sin medida, 1 intramural, 2 domiciliaria):")
print(resultado["TipoMedida"].value_counts().sort_index().to_string())

print("\nBoleta de salida:")
print(resultado["BoletaSalida"].value_counts().sort_index().to_string())

print("\nPreacuerdo:")
print(resultado["Preacuerdo"].value_counts().sort_index().to_string())

print("\nDelitos mas frecuentes (DelitoSentencia):")
print(resultado["DelitoSentencia"].value_counts().head(10).to_string())

print("\nSentencia condenatoria:")
print(resultado["Sentencia"].value_counts().to_string())

print("\nTipo de sentencia (0 sin sentencia, 1 condenatoria, 2 absolutoria):")
print(resultado["TipoSentencia"].value_counts().sort_index().to_string())

resultado.tail(50)

In [ ]:
def revisar_radicado(rad: str, ancho: int = 160) -> None:
    filas = base[base[COL_RADICADO].astype(str) == str(rad)].sort_values(COL_FECHA)

    if filas.empty:
        print(f"El radicado {rad} no está en el subconjunto de trabajo.")
        return

    print(f"=== {rad} — {len(filas)} actuaciones ===\n")

    for _, r in filas.iterrows():
        flags = [
            n.replace("flag_", "")
            for n in filas.columns
            if n.startswith("flag_") and r[n]
        ]
        print(f"[{r[COL_FECHA]:%Y-%m-%d}] flags: {', '.join(flags) or '—'}")
        print(f"    texto : {r['texto'][:ancho]}")
        print(f"    juzgado: {extraer_juzgado_serie(pd.Series([r['texto']]))[0]}")
        print(f"    delito : {clasificar_delitos(limpiar_texto(r['texto']))}")
        print()

    print("--- fila consolidada ---")
    print(
        resultado[resultado[COL_RADICADO].astype(str) == str(rad)]
        .T.to_string()
    )

In [ ]:
resultado.to_parquet("resultado_variables.parquet", index=False)